# Notebook 038: Delta OPTIMIZE, File Compaction and Auto-Compaction Strategies

**Module:** Module 04: Delta Lake Architecture, ACID Operations & Time Travel  
**Difficulty:** `Intermediate`  
**Databricks Masterclass Reference:** Video 13: Databricks Clusters & Video 23: Databricks Interview Questions  

---

## 1. Business Scenario & Real-World Context
A streaming ingestion job writes 1,000 tiny 50KB Parquet files into a Delta table every hour. Query latency has plummeted from 2 seconds to 45 seconds due to the classic Big Data 'small file problem' overwhelming driver filesystem metadata operations.

## 2. Technical Requirement & Problem Statement
Resolve small file degradation using Delta Lake compaction: execute `OPTIMIZE` command to bin-pack small files into standard 1GB target files, measure file count reduction, and configure Databricks auto-compaction and optimized writes (`spark.databricks.delta.optimizeWrite.enabled`).

## 3. Spark & Databricks Architecture Notes
The small file problem causes high metadata lookup overhead on cloud storage (millions of GET/LIST API calls). `OPTIMIZE` reads groups of tiny files and rewrites them into consolidated 1GB (or user-tuned size) Parquet files, creating a single commit in `_delta_log`.


## 4. Environment Setup & SparkSession Initialization
We configure a resilient `SparkSession` compatible with local PySpark development, Databricks runtime, or CI/CD test runners.

In [ ]:
import os
import sys
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import types as T
from pyspark.sql.window import Window

# Initialize SparkSession with Delta Lake support and optimized local configs
spark = (
    SparkSession.builder
    .appName(f"UseCase_038_Delta_OPTIMIZE,_File_Compaction_and_Auto-Compaction_Strategies")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "4")
    .config("spark.default.parallelism", "4")
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
print(f" Spark Version: {spark.version}")
print(f" Session Active: {spark.sparkContext.appName}")


## 5. Simulating Small File Ingestion Sprawl
Create 20 tiny files across multiple append operations.

In [ ]:
delta_opt_path = "/tmp/delta_optimize_038"
if Path(delta_opt_path).exists():
    shutil.rmtree(delta_opt_path)

# Append 20 micro-batches
for i in range(20):
    batch = [(f"ITEM-{i}-{j}", f"desc_{j}", 10.0 * j) for j in range(5)]
    spark.createDataFrame(batch, ["item_id", "desc", "val"]).write.format("delta").mode("append").save(delta_opt_path)

files_before = list(Path(delta_opt_path).glob("*.parquet"))
print(f"Total small Parquet files created before OPTIMIZE: {len(files_before)}")

## 6. Executing OPTIMIZE Bin-Packing Compaction
Run OPTIMIZE to compact small files into fewer large files.

In [ ]:
df_opt_metrics = spark.sql(f"OPTIMIZE delta.`{delta_opt_path}`")
df_opt_metrics.select("path", "metrics.numFilesAdded", "metrics.numFilesRemoved", "metrics.totalPushedFiles").show(truncate=False)

files_after = list(Path(delta_opt_path).glob("*.parquet"))
print(f"Total Parquet files on disk after OPTIMIZE: {len(files_after)} (Compacted down to active files!)")

## 7. Configuring Auto-Compaction & Optimized Writes
Enable automatic compaction at write time to prevent small file accumulation.

In [ ]:
# Enable Optimized Writes: Coalesces partitions dynamically before writing to disk
spark.conf.set("spark.databricks.delta.optimizeWrite.enabled", "true")

# Enable Auto Compaction: Automatically runs lightweight bin-packing after successful writes
spark.conf.set("spark.databricks.delta.autoCompact.enabled", "true")

print("Optimized Writes and Auto-Compaction enabled in SparkSession!")

## 9. Key Architectural Takeaways
- **`OPTIMIZE` bin-packs small files into target ~1GB files without altering data semantics.**
- **Optimized Writes (`delta.optimizeWrite`) coalesce small partitions during write operations, preventing fragmentation at the source.**
- **Auto-compaction (`delta.autoCompact`) automatically compacts small files right after commit, ideal for streaming ingest.**

## 10. Senior Data Engineering Interview Questions
1. What is the Small File Problem and how does it negatively affect query performance and cloud storage costs?
2. What is the difference between `autoCompact` and manually scheduled `OPTIMIZE` jobs?
3. Can users continue querying a Delta table while an `OPTIMIZE` command is actively running?

---
*Created for the **Spark Coding Use Cases Masterclass** repository referencing the Databricks All-in-One Playlist.*
